# Adding Local Time to TEMPO NO2

Converts each grid cell's UTC observation time to local time, using `timezonefinder` to look up the correct timezone per grid cell.

In [1]:
import pandas as pd
import numpy as np
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo

In [2]:
no2_file = '/global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/lowCF_TEMPO_03-07_0.1x0.1GCF_V04_MCD12Q1mask_2024.parquet'
no2_df = pd.read_parquet(no2_file, engine='pyarrow')

no2_df['datetime'] = pd.to_datetime(no2_df['datetime']).dt.floor('h')

no2_df['_wm_no2_mean'] = no2_df['no2_mean'] * no2_df['no2_count']
no2_df['_wm_no2_std'] = no2_df['no2_std'] * no2_df['no2_count']

agg = no2_df.groupby(['grid_lat', 'grid_lon', 'datetime'], as_index=False).agg(
    _wm_no2_mean_sum=('_wm_no2_mean', 'sum'),
    _wm_no2_std_sum=('_wm_no2_std', 'sum'),
    no2_count=('no2_count', 'sum'),
    eff_cloud_fraction_mean=('eff_cloud_fraction_mean', 'mean'),
    eff_cloud_fraction_std=('eff_cloud_fraction_std', 'mean'),
    amf_cloud_fraction_mean=('amf_cloud_fraction_mean', 'mean'),
    amf_cloud_fraction_std=('amf_cloud_fraction_std', 'mean'),
    snow_ice_fraction_mean=('snow_ice_fraction_mean', 'mean'),
    snow_ice_fraction_std=('snow_ice_fraction_std', 'mean'),)

agg['no2_mean'] = agg['_wm_no2_mean_sum'] / agg['no2_count']
agg['no2_std'] = agg['_wm_no2_std_sum'] / agg['no2_count']
agg = agg.drop(columns=['_wm_no2_mean_sum', '_wm_no2_std_sum'])

no2_df = agg
no2_df['no2_sem'] = no2_df['no2_std'] / np.sqrt(no2_df['no2_count'])
no2_df.loc[no2_df['no2_count'] <= 1, 'no2_sem'] = np.nan
no2_df['cv_no2'] = no2_df['no2_std'] / no2_df['no2_mean']

print(f'Rows after true hourly aggregation: {len(no2_df):,}')
no2_df.head()

Rows after true hourly aggregation: 7,506,097


,grid_lat,grid_lon,datetime,no2_count,eff_cloud_fraction_mean,eff_cloud_fraction_std,amf_cloud_fraction_mean,amf_cloud_fraction_std,snow_ice_fraction_mean,snow_ice_fraction_std,no2_mean,no2_std,no2_sem,cv_no2
0,36.05,-102.85,2024-03-01 15:00:00,23,0.000038,0.000123,0.000124,0.000406,0.0,0.0,5.107637e+14,2.881142e+14,6.007596e+13,0.564085
1,36.05,-102.85,2024-03-01 16:00:00,25,0.000386,0.000810,0.001339,0.002804,0.0,0.0,4.682882e+14,2.243396e+14,4.486792e+13,0.479063
2,36.05,-102.85,2024-03-01 17:00:00,21,0.020986,0.007428,0.070580,0.024170,0.0,0.0,3.129898e+14,1.499473e+14,3.272118e+13,0.479080
3,36.05,-102.85,2024-03-01 18:00:00,16,0.002185,0.002678,0.008136,0.009931,0.0,0.0,6.396807e+14,4.136407e+14,1.034102e+14,0.646636
4,36.05,-102.85,2024-03-01 19:00:00,21,0.000956,0.002153,0.003742,0.008404,0.0,0.0,4.261231e+14,3.299356e+14,7.199785e+13,0.774273


## Look up timezone per grid cell, convert to local time

In [3]:
tf = TimezoneFinder()

unique_coords = no2_df[['grid_lat', 'grid_lon']].drop_duplicates().copy()
unique_coords['tz_name'] = unique_coords.apply( 
    lambda row: tf.timezone_at(lat=row['grid_lat'], lng=row['grid_lon']), axis=1)

print(f'Unique grid cells: {len(unique_coords)}') 
print(unique_coords['tz_name'].value_counts()) 

no2_df = no2_df.merge(unique_coords, on=['grid_lat', 'grid_lon'], how='left')
no2_df['datetime_utc'] = pd.to_datetime(no2_df['datetime']) # makes copy

Unique grid cells: 10995
tz_name
America/Chicago                   9049
America/Indiana/Indianapolis       535
America/New_York                   490
America/Toronto                    306
America/Detroit                    303
America/Denver                     240
America/Indiana/Vincennes           33
America/North_Dakota/New_Salem      16
America/Indiana/Winamac             12
America/Indiana/Knox                 6
America/North_Dakota/Beulah          3
America/Indiana/Petersburg           2
Name: count, dtype: int64


In [4]:
local_times = pd.Series(index=no2_df.index, dtype='datetime64[ns]') # empty series 

for tz_name, group_idx in no2_df.groupby('tz_name').groups.items():
    tz = ZoneInfo(tz_name)
    utc_times = no2_df.loc[group_idx, 'datetime_utc'].dt.tz_localize('UTC')
    local_times.loc[group_idx] = utc_times.dt.tz_convert(tz).dt.tz_localize(None)

no2_df['datetime_local'] = local_times
no2_df['local_hour'] = no2_df['datetime_local'].dt.hour.astype('int64')

print(no2_df[['datetime_utc', 'grid_lat', 'grid_lon', 'tz_name', 'datetime_local', 'local_hour']]
      .head(10).to_string(index=False))

       datetime_utc  grid_lat  grid_lon         tz_name      datetime_local  local_hour
2024-03-01 15:00:00     36.05   -102.85 America/Chicago 2024-03-01 09:00:00           9
2024-03-01 16:00:00     36.05   -102.85 America/Chicago 2024-03-01 10:00:00          10
2024-03-01 17:00:00     36.05   -102.85 America/Chicago 2024-03-01 11:00:00          11
2024-03-01 18:00:00     36.05   -102.85 America/Chicago 2024-03-01 12:00:00          12
2024-03-01 19:00:00     36.05   -102.85 America/Chicago 2024-03-01 13:00:00          13
2024-03-01 20:00:00     36.05   -102.85 America/Chicago 2024-03-01 14:00:00          14
2024-03-01 21:00:00     36.05   -102.85 America/Chicago 2024-03-01 15:00:00          15
2024-03-01 22:00:00     36.05   -102.85 America/Chicago 2024-03-01 16:00:00          16
2024-03-02 15:00:00     36.05   -102.85 America/Chicago 2024-03-02 09:00:00           9
2024-03-02 16:00:00     36.05   -102.85 America/Chicago 2024-03-02 10:00:00          10


In [5]:
from pathlib import Path

output_file = '/global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/lowCF_TEMPO_03-07_0.1x0.1GCF_V04_MCD12Q1mask_2024_localtime.parquet'


Path(output_file).parent.mkdir(parents=True, exist_ok=True)
no2_df.to_parquet(output_file, engine='pyarrow', index=False)
print('Saved to:', output_file)


Saved to: /global/scratch/users/rubypercival/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/lowCF_TEMPO_03-07_0.1x0.1GCF_V04_MCD12Q1mask_2024_localtime.parquet
